***Small Language Models -- Part 1 -- Fundamentals***

*Based on a notebook originally written by Pratheerth Padman*

In [ ]:
# Load Hugging Face Token from shared environment
# 
from dotenv import load_dotenv
import os
load_dotenv('../shared/.env')
key = os.getenv('HF_TOKEN')
os.environ["HF_TOKEN"] = key

In [ ]:
# install required libraries  
# This is not on the image so we need to install


!pip install accelerate>=0.26.0 --quiet
print("Libraries installed!")

In [ ]:
# Now we import the packages needed.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Check to make certain we have a GPU
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))



In [ ]:
import time

In [ ]:
#def get_gpu_mem_mb():
    # Returns currently allocated GPU memory in MB
#    return torch.cuda.memory_allocated() / (1024 ** 2)

The Models have been downloaded and stored in a shared directory
* Llama-3.2-1B-Instruct
* gemma-2-2b-it
* Qwen/Qwen2.5-0.5B-Instruct

There are also quantized gguf versions of each.


In [ ]:
#base_mem = get_gpu_mem_mb()

In [ ]:
#print(f"Base Memory: {base_mem:.2f} MB")

In [ ]:
local_model_path = "/home/jovyan/shared/models/meta-llama--Llama-3.2-1B-Instruct"

tokenizer_llama = AutoTokenizer.from_pretrained(local_model_path)
t0 = time.time()
model_llama = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"meta-llama load took {time.time()-t0:.2f}s")
print("Llama 3.2 1B Instruct loaded")

In [ ]:
!nvidia-smi

In [ ]:
#mem_after_a = get_gpu_mem_mb()
#model_a_usage = mem_after_a - base_mem
#print(f"Model A Memory: {model_a_usage:.2f} MB")

In [ ]:
# llama verification
print("Running quick verification...")
_prompt = "Hi! How are you today?" # define a simple input string
# convert the prompt text to tokens, make it a pytorch tensor,
# and send it to the same device the model is on (the GPU)
_inputs = tokenizer_llama(_prompt, return_tensors="pt").to(model_llama.device)

# ask the model to generate a response based on the input
# max_new_tokens limits the output length to 5 tokens for quick check
_outputs = model_llama.generate(**_inputs, max_new_tokens=5)

# decode the numeric output tokens back into readable text
# outputs[0] ges the first sequence in the batch
print("Llama Verification Ok:", tokenizer_llama.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

In [ ]:
!nvidia-smi

In [ ]:
#Now load guff and quantized version
from llama_cpp import Llama

read_path = "/home/jovyan/shared/models/gguf/Llama-3.2-1B-Instruct-Q8_0.gguf"

t0 = time.time()
llmlamaGuff = Llama(model_path=read_path, n_gpu_layers=-1, verbose=False)
print(f"GGUF load took {time.time()-t0:.2f}s")

In [ ]:
t0 = time.time()
output = llmlamaGuff.create_chat_completion(
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=50,
)
print(f"Inference took {time.time()-t0:.2f}s")
print(output["choices"][0]["message"]["content"])

In [15]:
#mem_after_b = get_gpu_mem_mb()
#model_b_usage = mem_after_b - mem_after_a
#print(f"Model B Memory: {model_b_usage:.2f} MB")

In [16]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Thu Sep 17 02:42:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P0             29W /   70W |    7061MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
# --- Load Gemma 2 2B ---
local_model_path = "/home/jovyan/shared/models/google--gemma-2-2b-it"

tokenizer_gemma = AutoTokenizer.from_pretrained(local_model_path)
t0 = time.time()
model_gemma = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"Gemmaload took {time.time()-t0:.2f}s")
print(f"Gemma 2 2B loaded")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Gemmaload took 282.15s
Gemma 2 2B loaded


In [18]:
# gemini verification
print("Running quick verification...")
_prompt = "Hi! How are you today?"
_inputs = tokenizer_gemma(_prompt, return_tensors="pt").to(model_gemma.device)
_outputs = model_gemma.generate(**_inputs, max_new_tokens=5)

print("Gemma Verification Ok:", tokenizer_gemma.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

Running quick verification...
Gemma Verification Ok: <bos>Hi! How are you today? 😊

I'm


In [19]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Thu Sep 17 02:48:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P0             33W /   70W |   12243MiB /  15360MiB |     32%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
# =====  Now trty gguf
from llama_cpp import Llama

read_path = "/home/jovyan/shared/models/gguf/gemma-2-2b-it.q4_k_m.gguf"

t0 = time.time()
llmGemG = Llama(model_path=read_path, n_gpu_layers=-1, verbose =False)
print(f"GGUF load took {time.time()-t0:.2f}s")

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


GGUF load took 87.91s


In [21]:
t0 = time.time()
output = llmGemG.create_chat_completion(
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=50,
)
print(f"Inference took {time.time()-t0:.2f}s")
print(output["choices"][0]["message"]["content"])

Inference took 0.18s
Hello! 



In [22]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Thu Sep 17 02:50:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P0             29W /   70W |   14435MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
# --- Load Qwen 2.5 0.5B ---
local_model_path = "/home/jovyan/shared/models/Qwen--Qwen2.5-0.5B-Instruct"
t0 = time.time()
tokenizer_qwen = AutoTokenizer.from_pretrained(local_model_path)
model_qwen = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Qwenload took {time.time()-t0:.2f}s")
print("Qwen 2.5 0.5B loaded")

# qwen verification
print("Running quick verification...")
_prompt = "Hi! How are you today?"
_inputs = tokenizer_qwen(_prompt, return_tensors="pt").to(model_qwen.device)
_outputs = model_qwen.generate(**_inputs, max_new_tokens=5)
print("Qwen Verification Ok:", tokenizer_qwen.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

Some parameters are on the meta device because they were offloaded to the cpu.


Qwenload took 24.45s
Qwen 2.5 0.5B loaded
Running quick verification...
Qwen Verification Ok: Hi! How are you today? I'm really enjoying this


In [24]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Thu Sep 17 02:52:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             33W /   70W |   14821MiB /  15360MiB |     33%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [34]:
# ==== Now tryguff and quantized version
from llama_cpp import Llama

read_path = "/home/jovyan/shared/models/gguf/qwen2.5-0.5b-instruct-q8_0.gguf"

t0 = time.time()
llmQuenG = Llama(model_path=read_path, n_gpu_layers=-1, verbose=False)
print(f"GGUF load took {time.time()-t0:.2f}s")

llama_model_loader: loaded meta data with 26 key-value pairs and 291 tensors from /home/jovyan/shared/models/gguf/qwen2.5-0.5b-instruct-q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = qwen2.5-0.5b-instruct
llama_model_loader: - kv   3:                            general.version str              = v0.1
llama_model_loader: - kv   4:                           general.finetune str              = qwen2.5-0.5b-instruct
llama_model_loader: - kv   5:                         general.size_label str              = 630M
llama_model_loader: - kv   6:                          qwen2.block_count u32              = 24
llama_model_load

ValueError: Failed to load model from file: /home/jovyan/shared/models/gguf/qwen2.5-0.5b-instruct-q8_0.gguf

In [ ]:
t0 = time.time()
output = llmQuenG.create_chat_completion(
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=50,
)
print(f"Inference took {time.time()-t0:.2f}s")
print(output["choices"][0]["message"]["content"])

In [ ]:
!nvidia-smi

In [27]:
import time # Import the time library

# Define a function to handle text generation and timing
def generate_text(model, tokenizer, prompt, max_new_tokens):

    start_time = time.time() # Record the time before generation starts

    # Prepare the input: tokenize, convert to PyTorch tensors, move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate text using the model
    outputs = model.generate(
        **inputs,                     # Pass the tokenized inputs
        max_new_tokens=max_new_tokens, # Set maximum number of new tokens to generate
    )
    end_time = time.time() # Record the time after generation finishes

    # Process the output
    output_ids = outputs[0] # Get the full sequence of token IDs
    input_token_len = inputs.input_ids.shape[1] # Find length of original input tokens
    generated_ids = output_ids[input_token_len:] # Isolate the newly generated token IDs
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True) # Convert generated IDs to text
    num_generated_tokens = len(generated_ids) # Count how many tokens were generated

    # Calculate and display performance
    duration = end_time - start_time
    tokens_per_sec = num_generated_tokens / duration
    print(f"Generated {num_generated_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens/sec)")

    # Print the final generated text
    print("Output:")
    print(generated_text)

    # Return the text and speed for potential later use
    return generated_text, tokens_per_sec

In [28]:
# Define a function to handle text generation and timing
def generate_text_gguf(model, prompt, max_new_tokens):

    start_time = time.time() # Record the time before generation starts

    
# Create the messages list (OpenAI-compatible format)
    messages = [
           {"role": "user", "content": prompt}
    ]

# Generate a response using create_chat_completion
    response = model.create_chat_completion(
       messages = messages,
       max_tokens = max_new_tokens
    )

# Extract and print the response
    print("Response:")
    print(response["choices"][0]["message"]["content"])
    # Generate text using the model
    output =  response["choices"][0]["message"]["content"]
    end_time = time.time() # Record the time after generation finishes

    # Process the output
    completion_tokens = response["usage"]["completion_tokens"]
    # Calculate and display performance
    duration = end_time - start_time
    tokens_per_sec = len(output) / completion_tokens
    print(f"Generated {completion_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens per sec)")

   
    # Return the text and speed for potential later use
    return output, tokens_per_sec

In [29]:
prompt = "Describe the year-round climate in Paris, France"

In [30]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt, max_new_tokens=500)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Generated 500 tokens in 13.11 seconds (38.14 tokens/sec)
Output:
.
Paris, the capital of France, is known for its unique and varied climate. The city's climate is influenced by its location on the Seine River and its proximity to the Atlantic Ocean. Here's a breakdown of the year-round climate in Paris:

**Spring (March to May)**

* Temperatures gradually warm up, with average highs ranging from 12°C (54°F) in March to 18°C (64°F) in May.
* Spring is a popular time to visit Paris, with mild weather and fewer tourists compared to the summer months.
* The city experiences a moderate amount of rainfall during this time, with an average of 12 rainy days per month.

**Summer (June to August)**

* Summer is the warmest season in Paris, with average highs ranging from 22°C (72°F) in June to 25°C (77°F) in August.
* The city experiences high levels of sunshine, with an average of 7 hours of direct sunlight per day.
* Summer is also the wettest season, with an average of 17 rainy days per month

In [31]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text_gguf(llmlamaGuff, prompt, max_new_tokens=500)

Response:
Paris, the capital of France, has a temperate oceanic climate, characterized by mild winters and cool summers. The city's climate is influenced by its location on the Seine River and the surrounding mountains, which create a microclimate with distinct seasonal variations.

**Seasonal Climate Patterns:**

1. **Winter (December to February):** Paris experiences a moderate climate during winter, with average temperatures ranging from 2°C (36°F) to 6°C (43°F) throughout the month. The city is known for its cold and snowy winters, with an average of 12 rainy days per month.
2. **Spring (March to May):** Spring in Paris is mild and pleasant, with temperatures gradually warming up to 15°C (59°F) to 20°C (68°F) during the month. The city experiences a moderate amount of rainfall, with an average of 12 rainy days per month.
3. **Summer (June to August):** Summer in Paris is warm and sunny, with average temperatures ranging from 18°C (64°F) to 23°C (73°F). The city experiences a modera

In [32]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt, max_new_tokens=500)

Generated 467 tokens in 29.25 seconds (15.96 tokens/sec)
Output:
.

Paris, France, enjoys a **temperate maritime climate**, which means it experiences mild, wet winters and warm, dry summers. 

Here's a breakdown:

**Winter (November - March):**
* **Temperature:** Average highs around 5°C (41°F), with lows around 0°C (32°F).
* **Precipitation:**  Significant rainfall, with an average of 100-150 mm (4-6 inches) per month.
* **Sunshine:**  Limited sunshine, with an average of 3-4 hours per day.

**Spring (April - May):**
* **Temperature:**  Gradually warming up, with average highs around 10°C (50°F).
* **Precipitation:**  Rainfall decreases, with an average of 50-70 mm (2-3 inches) per month.
* **Sunshine:**  Increasing sunshine, with an average of 6-8 hours per day.

**Summer (June - August):**
* **Temperature:**  Warm and sunny, with average highs around 25°C (77°F).
* **Precipitation:**  Very little rainfall, with an average of 20-30 mm (1-1.5 inches) per month.
* **Sunshine:**  Abund

In [ ]:
# Generate with Gemma 2 2B GGUF
gemma_output, gemma_speed = generate_text_gguf(llmGemG, prompt, max_new_tokens=500)

In [ ]:
# Generate with Qwen 2.5 0.5B
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt, max_new_tokens=500)

In [ ]:
# Generate with Qwen 2.5 0.5B GGUF
qwen_output, qwen_speed = generate_text_gguf(llmQuenG, prompt, max_new_tokens=500)

In [ ]:
prompt_2 = """Scenario:
Four colored cups – Red, Blue, Green, and Yellow – are arranged in a circle.
One cup has a hidden star.

Facts:

The cup with the star is not Red.
The cup with the star is directly next to the Blue cup.
The Green cup is directly between the Yellow cup and the Red cup (when going around the circle in one direction).
Question:
Which color cup has the star? Just give the correct answer"""

In [ ]:
prompt_2b = """Scenario:
There are three blocks. 
Block 1 is directly on the table.
Block 2 is directly on block 1.
Block 3 is directly  on block 2.

Facts:
Block 1 is blue.
Block  3 is green.
The middle block can be either any color.


Question:
Is there a green block directly ontop of a non-green block? Justify the answer."""

In [ ]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt_2, max_new_tokens=500)

In [ ]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt_2b, max_new_tokens=500)

In [ ]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt_2, max_new_tokens=500)

In [ ]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt_2b, max_new_tokens=500)

In [ ]:
# Generate with Qwen 2.5 0.5B
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt_2, max_new_tokens=500)

In [ ]:
# Generate with Qwen 2.5 0.5B GGUF
qwen_output, qwen_speed = generate_text_gguf(llmQuenG, prompt_2, max_new_tokens=500)

In [ ]:
prompt_3 = """Write a simple Python function called `calculate_area`
that accepts the length and width of a rectangle and returns the area."""

In [ ]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Llama 3.2 1B GGuf
llama_output, llama_speed = generate_text_gguf(llmlamaGuff, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text_gguf(llmGemG, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Qwen 2.5 0.5B
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Qwen 2.5 0.5B gguf
qwen_output, qwen_speed = generate_text_gguf(llmQuenG, prompt_3, max_new_tokens=500)